<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/Exercises_XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [ ]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)


## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')

# Remplir le test et la création de la colonne d'identifiants
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))  # Crée un ID unique pour chaque ligne

display(pdf.head())

# Créer un sous-ensemble gérable (les 1000 premières lignes)
pdf_subset = pdf.head(1000).copy()

# Afficher les colonnes 'id' et 'title' du sous-ensemble
display(pdf_subset[['id', 'title']].head())

## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [ ]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Todo: create training examples from the subset data using the example_create_fn
faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset.iterrows()
]

# Afficher les 2 premiers exemples pour vérification
faiss_train_examples[:2]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # 1. Encode the query and convert to a 2D float32 numpy array
    query_vector = np.array(model.encode([query])).astype('float32')

    # Normalize the query vector for cosine similarity
    faiss.normalize_L2(query_vector)

    # 2. Search the index using the query vector and the number of results (k)
    sims, ids = index_content.search(query_vector, k)

    # Filter the DataFrame to keep only the matching IDs
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()

    # Map the similarity scores back to the results
    results['similarities'] = sims[0]
    return results

# Test the function
display(search_content('animal', pdf_to_index, k=5))

## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

# To-Do: create the collection and add documents
collection = chroma_client.create_collection(name=collection_name)

# Add the first 100 documents, their metadata (topics), and string IDs
collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()]
)

# Query the collection for the top 10 most relevant documents
results = collection.query(
    query_texts=["space"],
    n_results=10
)

print(json.dumps(results, indent=2))

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [ ]:
model_id = 'google/flan-t5-small'  # lightweight, better than tiny GPT-2 for QA

# To-Do: create the text2text-generation pipeline
pipe = pipeline(
    "text2text-generation",
    model=model_id,
    max_new_tokens=512,  # Maximum length of the generated answer
    device_map="auto"    # Automatically uses GPU if available, otherwise CPU
)

question = "What's the latest news on space development?"
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"

response = pipe(prompt)[0]['generated_text']
print(response)